In [2]:
pip install rouge-score jiwer tqdm pandas gitpython

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b480be40407a8c3064180896272113a53702cc6742e40f3437f1c5335e9028d1
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [10]:
import os
import subprocess
import json
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer

# ==============================
# CONFIG
# ==============================
DATASET_REPO = "https://github.com/wyim/aci-bench.git"
LOCAL_DIR = "./aci_bench"
DATA_PATH = os.path.join(LOCAL_DIR, "data", "challenge_data_json")


# ==============================
# 1. DOWNLOAD DATASET
# ==============================
def download_dataset():
    if not os.path.exists(LOCAL_DIR):
        print("📥 Cloning dataset...")
        subprocess.run(["git", "clone", DATASET_REPO, LOCAL_DIR], check=True)
    else:
        print("✅ Dataset already exists. Skipping download.")


# ==============================
# 2. LOAD DATASET
# ==============================
def load_dataset():
    data = []

    print("📂 Loading dataset...")

    for file in os.listdir(DATA_PATH):
        if file.endswith(".json"):
            file_path = os.path.join(DATA_PATH, file)

            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = json.load(f)

                    # ✅ Flatten structure
                    if isinstance(content, dict) and "data" in content:
                        data.extend(content["data"])
                    elif isinstance(content, list):
                        data.extend(content)
                    else:
                        data.append(content)

            except Exception as e:
                print(f"[WARNING] Skipping {file}: {e}")

    if len(data) == 0:
        raise RuntimeError("No valid data loaded")

    print(f"✅ Loaded {len(data)} samples")
    return data


# ==============================
# 3. BASELINE SUMMARIZER
# ==============================
def baseline_summarizer(text):
    sentences = text.split(".")
    return ".".join(sentences[:3]).strip()


# ==============================
# 4. FIELD EXTRACTION
# ==============================
def extract_fields(sample):
    transcript = (
        sample.get("src")
        or sample.get("dialogue")
        or sample.get("transcript")
        or ""
    )

    reference = (
        sample.get("tgt")
        or sample.get("summary")
        or sample.get("note")
        or ""
    )

    return transcript, reference


# ==============================
# 5. SIMPLE MEDICAL ENTITY EXTRACTION
# ==============================
def extract_medical_entities(text):
    keywords = [
        "pain", "fever", "cough", "diabetes", "hypertension",
        "infection", "antibiotic", "treatment", "medication",
        "diagnosis", "symptom", "chest", "headache"
    ]

    text = text.lower()
    found = set()

    for k in keywords:
        if k in text:
            found.add(k)

    return found


# ==============================
# 6. EVALUATION
# ==============================
def evaluate(data):
    rouge = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    results = []

    for sample in tqdm(data, desc="Evaluating"):
        try:
            transcript, reference = extract_fields(sample)

            if not transcript or not reference:
                continue

            prediction = baseline_summarizer(transcript)

            rouge_scores = rouge.score(reference, prediction)

            # ✅ Clinical completeness
            ref_entities = extract_medical_entities(reference)
            pred_entities = extract_medical_entities(prediction)

            if len(ref_entities) > 0:
                completeness = len(pred_entities & ref_entities) / len(ref_entities)
            else:
                completeness = 1.0

            results.append({
                "rouge1": rouge_scores["rouge1"].fmeasure,
                "rouge2": rouge_scores["rouge2"].fmeasure,
                "rougeL": rouge_scores["rougeL"].fmeasure,
                "completeness": completeness
            })

        except Exception as e:
            print(f"[WARNING] Skipping sample: {e}")

    if len(results) == 0:
        raise RuntimeError("No valid evaluation results")

    return pd.DataFrame(results)


# ==============================
# 7. REPORT GENERATION
# ==============================
def generate_report(df):
    report = {
        "ROUGE-1": df["rouge1"].mean(),
        "ROUGE-2": df["rouge2"].mean(),
        "ROUGE-L": df["rougeL"].mean(),
        "Clinical Completeness": df["completeness"].mean()
    }

    print("\n===== 📊 BASELINE REPORT =====")
    for k, v in report.items():
        print(f"{k}: {v:.4f}")

    df.to_csv("evaluation_results.csv", index=False)

    with open("summary_report.json", "w") as f:
        json.dump(report, f, indent=4)

    print("\n✅ Files saved:")
    print("- evaluation_results.csv")
    print("- summary_report.json")


# ==============================
# MAIN
# ==============================
def main():
    download_dataset()
    data = load_dataset()

    print("\n🔍 Sample after fix:")
    print(data[0])

    df = evaluate(data)
    generate_report(df)


if __name__ == "__main__":
    main()

✅ Dataset already exists. Skipping download.
📂 Loading dataset...
✅ Loaded 1242 samples

🔍 Sample after fix:
{'src': "[doctor] hi , martha . how are you ?\n[patient] i'm doing okay . how are you ?\n[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?\n[patient] okay .\n[doctor] martha is a 50-year-old female with a past medical history significant for congestive heart failure , depression and hypertension who presents for her annual exam . so , martha , it's been a year since i've seen you . how are you doing ?\n[patient] i'm doing well . i've been traveling a lot recently since things have , have gotten a bit lighter . and i got my , my vaccine , so i feel safer about traveling . i've been doing a lot of hiking . uh , went to washington last weekend to hike in northern cascades, like around the mount baker area .\n[doctor] nice . that's great . i'm glad to hear that you're staying active , you know . i , i just love 

Evaluating: 100%|██████████| 1242/1242 [01:15<00:00, 16.50it/s]


===== 📊 BASELINE REPORT =====
ROUGE-1: 0.1142
ROUGE-2: 0.0389
ROUGE-L: 0.0708
Clinical Completeness: 0.5975

✅ Files saved:
- evaluation_results.csv
- summary_report.json
